In [1]:
"""
Batch evaluation of 15 GA+SMOTE models with resource monitoring

Author: Tan Yi Feng
Student ID: 23WMR14766
"""

import pandas as pd
import numpy as np
import joblib
import pickle
import time
import psutil
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_recall_fscore_support,
                             classification_report)
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from pathlib import Path
import sys


SCRIPT_DIR = Path.cwd()


DATA_PATH = SCRIPT_DIR / 'data' / 'FYP_Processed_Data' / 'balanced_train_data.csv'
MODELS_DIR = SCRIPT_DIR / 'data' / 'models'


MODEL_FILES = [
    (MODELS_DIR / 'best_rf_pipeline2.pkl', MODELS_DIR / 'selected_features2.pkl'),
    (MODELS_DIR / 'best_rf_pipeline3.pkl', MODELS_DIR / 'selected_features3.pkl'),
    (MODELS_DIR / 'best_rf_pipeline4.pkl', MODELS_DIR / 'selected_features4.pkl'),
    (MODELS_DIR / 'best_rf_pipeline5.pkl', MODELS_DIR / 'selected_features5.pkl'),
    (MODELS_DIR / 'best_rf_pipeline6.pkl', MODELS_DIR / 'selected_features6.pkl'),
    (MODELS_DIR / 'best_rf_pipeline7.pkl', MODELS_DIR / 'selected_features7.pkl'),
    (MODELS_DIR / 'best_rf_pipeline8.pkl', MODELS_DIR / 'selected_features8.pkl'),
    (MODELS_DIR / 'best_rf_pipeline9.pkl', MODELS_DIR / 'selected_features9.pkl'),
    (MODELS_DIR / 'best_rf_pipeline10.pkl', MODELS_DIR / 'selected_features10.pkl'),
    (MODELS_DIR / 'best_rf_pipeline11.pkl', MODELS_DIR / 'selected_features11.pkl'),
    (MODELS_DIR / 'best_rf_pipeline12.pkl', MODELS_DIR / 'selected_features12.pkl'),
    (MODELS_DIR / 'best_rf_pipeline13.pkl', MODELS_DIR / 'selected_features13.pkl'),
    (MODELS_DIR / 'best_rf_pipeline14.pkl', MODELS_DIR / 'selected_features14.pkl'),
    (MODELS_DIR / 'best_rf_pipeline15.pkl', MODELS_DIR / 'selected_features15.pkl'),
    (MODELS_DIR / 'best_rf_pipeline16.pkl', MODELS_DIR / 'selected_features16.pkl'),
]


label_merge_map = {
    1: 1, 2: 2, 3: 3, 4: 2, 5: 3,
    6: 4, 7: 5, 8: 6, 9: 5, 10: 5,
    11: 7, 15: 7, 12: 8, 16: 8,
    13: 9, 17: 9, 14: 10, 18: 10
}


CLASS_NAMES = {
    0: 'Class_0', 1: 'Class_1', 2: 'Class_2', 3: 'Class_3',
    4: 'Class_4', 5: 'Class_5', 6: 'Class_6', 7: 'Class_7',
    8: 'Class_8', 9: 'Class_9',
}



def print_run_report(run_num, n_test, n_features, inference_time,
                     mem_used, cpu_usage, acc, f1_macro, f1_weighted,
                     y_test, y_pred, label_encoder):
    """Print the formatted output block for a single run."""

    avg_time_per_sample = inference_time / n_test

   
    classes = label_encoder.classes_          
    target_names = [CLASS_NAMES.get(c, f'Class_{c}') for c in classes]

    report = classification_report(
        y_test, y_pred,
        target_names=target_names,
        zero_division=0,
        digits=4
    )

    w = 72
    print("\n" + "=" * w)
    print(f"  RUN {run_num:>2}  —  GA+SMOTE RESOURCE & PERFORMANCE")
    print("=" * w)
    print(f"  {'Test Set Size':<28}: {n_test:,} samples")
    print(f"  {'Features Used':<28}: {n_features}")
    print(f"  {'Total Inference Time':<28}: {inference_time:.4f} s")
    print(f"  {'Avg Time per Sample':<28}: {avg_time_per_sample:.8f} s")
    print(f"  {'Peak Memory Usage':<28}: {mem_used:.2f} MB")
    print(f"  {'CPU Utilization':<28}: {cpu_usage:.2f} %")
    print("-" * w)
    print(f"  {'Accuracy':<28}: {acc:.4f}")
    print(f"  {'Macro F1-Score':<28}: {f1_macro:.4f}")
    print(f"  {'Weighted F1-Score':<28}: {f1_weighted:.4f}")
    print("=" * w)
    print("  DETAILED CLASSIFICATION REPORT")
    print("=" * w)
 
    for line in report.splitlines():
        print("  " + line)
    print("=" * w)



print("=" * 80)
print("BATCH EVALUATION: 15 GA+SMOTE MODELS WITH RESOURCE MONITORING")
print("=" * 80)

print("\n[1] Loading dataset...")
df = pd.read_csv(DATA_PATH)
df = df[df['label'] != 0].copy()
df['label'] = df['label'].map(label_merge_map)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
X = df.drop(columns=['label']).select_dtypes(include=['number'])
y = df['label']

X = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X), columns=X.columns)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"    Test size: {len(X_test):,}")


results = []
process = psutil.Process(os.getpid())

for run_num, (model_path, features_path) in enumerate(MODEL_FILES, start=1):
    print(f"\n>>> Loading Run {run_num} ({Path(model_path).name}) ...")

    try:
        # Load model + selected features
        model = joblib.load(model_path)
        with open(features_path, 'rb') as f:
            selected_features = list(pickle.load(f))

        X_test_sel = X_test[selected_features]

       
        mem_before = process.memory_info().rss / (1024 ** 2)
        cpu_before = psutil.cpu_percent(interval=None)   

        inference_start = time.perf_counter()
        y_pred = model.predict(X_test_sel)
        inference_time = time.perf_counter() - inference_start

        mem_after  = process.memory_info().rss / (1024 ** 2)
        mem_used   = max(mem_after - mem_before, 0.0)   
        cpu_usage  = psutil.cpu_percent(interval=None)

       
        acc         = accuracy_score(y_test, y_pred)
        f1_macro    = f1_score(y_test, y_pred, average='macro',    zero_division=0)
        f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        precision, recall, _, _ = precision_recall_fscore_support(
            y_test, y_pred, average=None, zero_division=0
        )
        weights            = np.bincount(y_test)
        precision_weighted = np.average(precision, weights=weights)
        recall_weighted    = np.average(recall,    weights=weights)

       
        print_run_report(
            run_num       = run_num,
            n_test        = len(X_test),
            n_features    = len(selected_features),
            inference_time= inference_time,
            mem_used      = mem_used,
            cpu_usage     = cpu_usage,
            acc           = acc,
            f1_macro      = f1_macro,
            f1_weighted   = f1_weighted,
            y_test        = y_test,
            y_pred        = y_pred,
            label_encoder = label_encoder,
        )

        results.append({
            'Run':               run_num,
            'Features':          len(selected_features),
            'Accuracy':          acc,
            'Precision (W)':     precision_weighted,
            'Recall (W)':        recall_weighted,
            'F1 (Macro)':        f1_macro,
            'F1 (Weighted)':     f1_weighted,
            'Inference Time (s)':inference_time,
            'Avg Time/Sample (s)':inference_time / len(X_test),
            'Memory (MB)':       mem_used,
            'CPU (%)':           cpu_usage,
        })

    except Exception as e:
        print(f"    ❌ Error in Run {run_num}: {e}")
        continue


if not results:
    print("\n❌ No models evaluated!")
    exit()

results_df = pd.DataFrame(results)

print("\n" + "=" * 80)
print("GA OPTIMIZATION STABILITY (15 RUNS) — SUMMARY TABLE")
print("=" * 80)
print(results_df.to_string(index=False,
      float_format=lambda x: f'{x:.4f}' if abs(x) < 1000 else f'{x:.2f}'))

print("\n" + "=" * 80)
print("AGGREGATE STATISTICS")
print("=" * 80)
print(f"{'Metric':<30} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 80)

for col in results_df.columns[1:]:
    m, s, lo, hi = (results_df[col].mean(), results_df[col].std(),
                    results_df[col].min(),  results_df[col].max())
    fmt = '.1f' if col == 'Features' else ('.4f' if m < 1000 else '.2f')
    print(f"{col:<30} {m:>10{fmt}} {s:>10{fmt}} {lo:>10{fmt}} {hi:>10{fmt}}")

print("\n" + "=" * 80)
print("COEFFICIENT OF VARIATION  (Lower = More Stable)")
print("=" * 80)
for col in ['Accuracy', 'F1 (Weighted)', 'F1 (Macro)']:
    cv = (results_df[col].std() / results_df[col].mean()) * 100
    print(f"  {col:<28} {cv:.4f}%")
print("=" * 80)

# Save
output_file = 'ga_stability_results_with_resources.csv'
results_df.to_csv(output_file, index=False)
print(f"\n💾 Results saved to: {output_file}")
print("\n✅ Batch evaluation complete!")

BATCH EVALUATION: 15 GA+SMOTE MODELS WITH RESOURCE MONITORING

[1] Loading dataset...
    Test size: 3,952

>>> Loading Run 1 (best_rf_pipeline2.pkl) ...

  RUN  1  —  GA+SMOTE RESOURCE & PERFORMANCE
  Test Set Size               : 3,952 samples
  Features Used               : 25
  Total Inference Time        : 0.0736 s
  Avg Time per Sample         : 0.00001862 s
  Peak Memory Usage           : 1.09 MB
  CPU Utilization             : 67.70 %
------------------------------------------------------------------------
  Accuracy                    : 0.9691
  Macro F1-Score              : 0.9318
  Weighted F1-Score           : 0.9693
  DETAILED CLASSIFICATION REPORT
                precision    recall  f1-score   support
  
       Class_1     0.6987    0.6813    0.6899       160
       Class_2     1.0000    0.9938    0.9969       320
       Class_3     0.9969    1.0000    0.9984       320
       Class_4     0.8068    0.8875    0.8452       160
       Class_5     0.9630    0.9208    0.9414  

In [2]:
"""
Evaluation of Baseline RF model (no GA feature selection)
Mirrors the GA+SMOTE evaluation format exactly

Author: Tan Yi Feng
Student ID: 23WMR14766
"""

import pandas as pd
import numpy as np
import joblib
import time
import psutil
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_recall_fscore_support,
                             classification_report)
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

from pathlib import Path
import sys


SCRIPT_DIR = Path.cwd()


DATA_PATH = SCRIPT_DIR / 'data' / 'FYP_Processed_Data' / 'balanced_train_data.csv'
MODELS_DIR = SCRIPT_DIR / 'data' / 'models'


MODEL_PATH = MODELS_DIR /'baseline_rf_model.pkl'


label_merge_map = {
    1: 1, 2: 2, 3: 3, 4: 2, 5: 3,
    6: 4, 7: 5, 8: 6, 9: 5, 10: 5,
    11: 7, 15: 7, 12: 8, 16: 8,
    13: 9, 17: 9, 14: 10, 18: 10
}


CLASS_NAMES = {
    0: 'Class_0', 1: 'Class_1', 2: 'Class_2', 3: 'Class_3',
    4: 'Class_4', 5: 'Class_5', 6: 'Class_6', 7: 'Class_7',
    8: 'Class_8', 9: 'Class_9',
}



def print_baseline_report(n_test, n_features, inference_time,
                          mem_used, cpu_usage, acc, f1_macro, f1_weighted,
                          y_test, y_pred, label_encoder):

    avg_time_per_sample = inference_time / n_test
    classes      = label_encoder.classes_
    target_names = [CLASS_NAMES.get(c, f'Class_{c}') for c in classes]

    report = classification_report(
        y_test, y_pred,
        target_names=target_names,
        zero_division=0,
        digits=4
    )

    w = 72
    print("\n" + "=" * w)
    print("  BASELINE MODEL  —  RESOURCE & PERFORMANCE")
    print("=" * w)
    print(f"  {'Test Set Size':<28}: {n_test:,} samples")
    print(f"  {'Features Used':<28}: {n_features}  (all — no GA)")
    print(f"  {'Total Inference Time':<28}: {inference_time:.4f} s")
    print(f"  {'Avg Time per Sample':<28}: {avg_time_per_sample:.8f} s")
    print(f"  {'Peak Memory Usage':<28}: {mem_used:.2f} MB")
    print(f"  {'CPU Utilization':<28}: {cpu_usage:.2f} %")
    print("-" * w)
    print(f"  {'Accuracy':<28}: {acc:.4f}")
    print(f"  {'Macro F1-Score':<28}: {f1_macro:.4f}")
    print(f"  {'Weighted F1-Score':<28}: {f1_weighted:.4f}")
    print("=" * w)
    print("  DETAILED CLASSIFICATION REPORT")
    print("=" * w)
    for line in report.splitlines():
        print("  " + line)
    print("=" * w)



print("=" * 80)
print("BASELINE MODEL EVALUATION")
print("=" * 80)

print("\n[1] Loading dataset...")
df = pd.read_csv(DATA_PATH)
df = df[df['label'] != 0].copy()
df['label'] = df['label'].map(label_merge_map)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
X = df.drop(columns=['label']).select_dtypes(include=['number'])
y = df['label']

X = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X), columns=X.columns)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"    Test size  : {len(X_test):,}")
print(f"    Features   : {X_test.shape[1]} (full feature set)")


print("\n[2] Loading baseline model...")
model = joblib.load(MODEL_PATH)

process = psutil.Process(os.getpid())

mem_before  = process.memory_info().rss / (1024 ** 2)
_           = psutil.cpu_percent(interval=None)  

inference_start = time.perf_counter()
y_pred          = model.predict(X_test)
inference_time  = time.perf_counter() - inference_start

mem_after  = process.memory_info().rss / (1024 ** 2)
mem_used   = max(mem_after - mem_before, 0.0)
cpu_usage  = psutil.cpu_percent(interval=None)


acc         = accuracy_score(y_test, y_pred)
f1_macro    = f1_score(y_test, y_pred, average='macro',    zero_division=0)
f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

precision, recall, _, _ = precision_recall_fscore_support(
    y_test, y_pred, average=None, zero_division=0
)
weights            = np.bincount(y_test)
precision_weighted = np.average(precision, weights=weights)
recall_weighted    = np.average(recall,    weights=weights)


print_baseline_report(
    n_test        = len(X_test),
    n_features    = X_test.shape[1],
    inference_time= inference_time,
    mem_used      = mem_used,
    cpu_usage     = cpu_usage,
    acc           = acc,
    f1_macro      = f1_macro,
    f1_weighted   = f1_weighted,
    y_test        = y_test,
    y_pred        = y_pred,
    label_encoder = label_encoder,
)


results = {
    'Model':             'Baseline (No GA)',
    'Features':          X_test.shape[1],
    'Accuracy':          acc,
    'Precision (W)':     precision_weighted,
    'Recall (W)':        recall_weighted,
    'F1 (Macro)':        f1_macro,
    'F1 (Weighted)':     f1_weighted,
    'Inference Time (s)':inference_time,
    'Avg Time/Sample (s)':inference_time / len(X_test),
    'Memory (MB)':       mem_used,
    'CPU (%)':           cpu_usage,
}

pd.DataFrame([results]).to_csv('baseline_eval_results.csv', index=False)
print("\n💾 Results saved to: baseline_eval_results.csv")
print("\n✅ Baseline evaluation complete!")

BASELINE MODEL EVALUATION

[1] Loading dataset...
    Test size  : 3,952
    Features   : 39 (full feature set)

[2] Loading baseline model...

  BASELINE MODEL  —  RESOURCE & PERFORMANCE
  Test Set Size               : 3,952 samples
  Features Used               : 39  (all — no GA)
  Total Inference Time        : 0.0925 s
  Avg Time per Sample         : 0.00002341 s
  Peak Memory Usage           : 0.00 MB
  CPU Utilization             : 54.50 %
------------------------------------------------------------------------
  Accuracy                    : 0.9747
  Macro F1-Score              : 0.9428
  Weighted F1-Score           : 0.9746
  DETAILED CLASSIFICATION REPORT
                precision    recall  f1-score   support
  
       Class_1     0.7682    0.7250    0.7460       160
       Class_2     1.0000    0.9938    0.9969       320
       Class_3     0.9969    1.0000    0.9984       320
       Class_4     0.8471    0.9000    0.8727       160
       Class_5     0.9580    0.9500    0.954

In [4]:
"""
Evaluation of Smote only RF model (no GA feature selection)
Mirrors the GA+SMOTE evaluation format exactly

Author: Tan Yi Feng
Student ID: 23WMR14766
"""

import pandas as pd
import numpy as np
import joblib
import time
import psutil
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_recall_fscore_support,
                             classification_report)
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer

from pathlib import Path
import sys


SCRIPT_DIR = Path.cwd()


DATA_PATH = SCRIPT_DIR / 'data' / 'FYP_Processed_Data' / 'balanced_train_data.csv'
MODELS_DIR = SCRIPT_DIR / 'data' / 'models'


MODEL_PATH = MODELS_DIR /'smote_rf_model_v1.pkl'


label_merge_map = {
    1: 1, 2: 2, 3: 3, 4: 2, 5: 3,
    6: 4, 7: 5, 8: 6, 9: 5, 10: 5,
    11: 7, 15: 7, 12: 8, 16: 8,
    13: 9, 17: 9, 14: 10, 18: 10
}


CLASS_NAMES = {
    0: 'Class_0', 1: 'Class_1', 2: 'Class_2', 3: 'Class_3',
    4: 'Class_4', 5: 'Class_5', 6: 'Class_6', 7: 'Class_7',
    8: 'Class_8', 9: 'Class_9',
}



def print_baseline_report(n_test, n_features, inference_time,
                          mem_used, cpu_usage, acc, f1_macro, f1_weighted,
                          y_test, y_pred, label_encoder):

    avg_time_per_sample = inference_time / n_test
    classes      = label_encoder.classes_
    target_names = [CLASS_NAMES.get(c, f'Class_{c}') for c in classes]

    report = classification_report(
        y_test, y_pred,
        target_names=target_names,
        zero_division=0,
        digits=4
    )

    w = 72
    print("\n" + "=" * w)
    print("  SMOTE MODEL  —  RESOURCE & PERFORMANCE")
    print("=" * w)
    print(f"  {'Test Set Size':<28}: {n_test:,} samples")
    print(f"  {'Features Used':<28}: {n_features}  (all — no GA)")
    print(f"  {'Total Inference Time':<28}: {inference_time:.4f} s")
    print(f"  {'Avg Time per Sample':<28}: {avg_time_per_sample:.8f} s")
    print(f"  {'Peak Memory Usage':<28}: {mem_used:.2f} MB")
    print(f"  {'CPU Utilization':<28}: {cpu_usage:.2f} %")
    print("-" * w)
    print(f"  {'Accuracy':<28}: {acc:.4f}")
    print(f"  {'Macro F1-Score':<28}: {f1_macro:.4f}")
    print(f"  {'Weighted F1-Score':<28}: {f1_weighted:.4f}")
    print("=" * w)
    print("  DETAILED CLASSIFICATION REPORT")
    print("=" * w)
    for line in report.splitlines():
        print("  " + line)
    print("=" * w)



print("=" * 80)
print("SMOTE MODEL EVALUATION")
print("=" * 80)

print("\n[1] Loading dataset...")
df = pd.read_csv(DATA_PATH)
df = df[df['label'] != 0].copy()
df['label'] = df['label'].map(label_merge_map)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
X = df.drop(columns=['label']).select_dtypes(include=['number'])
y = df['label']

X = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X), columns=X.columns)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"    Test size  : {len(X_test):,}")
print(f"    Features   : {X_test.shape[1]} (full feature set)")


print("\n[2] Loading SMOTE model...")
model = joblib.load(MODEL_PATH)

process = psutil.Process(os.getpid())

mem_before  = process.memory_info().rss / (1024 ** 2)
_           = psutil.cpu_percent(interval=None)   

inference_start = time.perf_counter()
y_pred          = model.predict(X_test)
inference_time  = time.perf_counter() - inference_start

mem_after  = process.memory_info().rss / (1024 ** 2)
mem_used   = max(mem_after - mem_before, 0.0)
cpu_usage  = psutil.cpu_percent(interval=None)


acc         = accuracy_score(y_test, y_pred)
f1_macro    = f1_score(y_test, y_pred, average='macro',    zero_division=0)
f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

precision, recall, _, _ = precision_recall_fscore_support(
    y_test, y_pred, average=None, zero_division=0
)
weights            = np.bincount(y_test)
precision_weighted = np.average(precision, weights=weights)
recall_weighted    = np.average(recall,    weights=weights)


print_baseline_report(
    n_test        = len(X_test),
    n_features    = X_test.shape[1],
    inference_time= inference_time,
    mem_used      = mem_used,
    cpu_usage     = cpu_usage,
    acc           = acc,
    f1_macro      = f1_macro,
    f1_weighted   = f1_weighted,
    y_test        = y_test,
    y_pred        = y_pred,
    label_encoder = label_encoder,
)


results = {
    'Model':             'SMOTE (No GA)',
    'Features':          X_test.shape[1],
    'Accuracy':          acc,
    'Precision (W)':     precision_weighted,
    'Recall (W)':        recall_weighted,
    'F1 (Macro)':        f1_macro,
    'F1 (Weighted)':     f1_weighted,
    'Inference Time (s)':inference_time,
    'Avg Time/Sample (s)':inference_time / len(X_test),
    'Memory (MB)':       mem_used,
    'CPU (%)':           cpu_usage,
}

pd.DataFrame([results]).to_csv('smote_eval_results.csv', index=False)
print("\n💾 Results saved to: baseline_eval_results.csv")
print("\n✅ Baseline evaluation complete!")

SMOTE MODEL EVALUATION

[1] Loading dataset...
    Test size  : 3,952
    Features   : 39 (full feature set)

[2] Loading SMOTE model...

  SMOTE MODEL  —  RESOURCE & PERFORMANCE
  Test Set Size               : 3,952 samples
  Features Used               : 39  (all — no GA)
  Total Inference Time        : 0.1212 s
  Avg Time per Sample         : 0.00003066 s
  Peak Memory Usage           : 5.30 MB
  CPU Utilization             : 49.50 %
------------------------------------------------------------------------
  Accuracy                    : 0.9717
  Macro F1-Score              : 0.9371
  Weighted F1-Score           : 0.9721
  DETAILED CLASSIFICATION REPORT
                precision    recall  f1-score   support
  
       Class_1     0.6923    0.7312    0.7112       160
       Class_2     1.0000    0.9938    0.9969       320
       Class_3     0.9969    0.9969    0.9969       320
       Class_4     0.8208    0.8875    0.8529       160
       Class_5     0.9739    0.9333    0.9532       4

In [5]:
"""
Evaluation of GA only RF model (no Smote)
Mirrors the GA+SMOTE evaluation format exactly

Author: Tan Yi Feng
Student ID: 23WMR14766
"""

import pandas as pd
import numpy as np
import joblib
import pickle
import time
import psutil
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_recall_fscore_support,
                             classification_report)
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from pathlib import Path

from pathlib import Path
import sys


SCRIPT_DIR = Path.cwd()


DATA_PATH = SCRIPT_DIR / 'data' / 'FYP_Processed_Data' / 'balanced_train_data.csv'
MODELS_DIR = SCRIPT_DIR / 'data' / 'models'


MODEL_FILES = [
    (MODELS_DIR / 'bestGAonly_rf_pipeline18.pkl', MODELS_DIR / 'GAonly_selected_features18.pkl')
]


label_merge_map = {
    1: 1, 2: 2, 3: 3, 4: 2, 5: 3,
    6: 4, 7: 5, 8: 6, 9: 5, 10: 5,
    11: 7, 15: 7, 12: 8, 16: 8,
    13: 9, 17: 9, 14: 10, 18: 10
}


CLASS_NAMES = {
    0: 'Class_0', 1: 'Class_1', 2: 'Class_2', 3: 'Class_3',
    4: 'Class_4', 5: 'Class_5', 6: 'Class_6', 7: 'Class_7',
    8: 'Class_8', 9: 'Class_9',
}



def print_run_report(run_num, n_test, n_features, inference_time,
                     mem_used, cpu_usage, acc, f1_macro, f1_weighted,
                     y_test, y_pred, label_encoder):
    """Print the formatted output block for a single run."""

    avg_time_per_sample = inference_time / n_test

    
    classes = label_encoder.classes_          
    target_names = [CLASS_NAMES.get(c, f'Class_{c}') for c in classes]

    report = classification_report(
        y_test, y_pred,
        target_names=target_names,
        zero_division=0,
        digits=4
    )

    w = 72
    print("\n" + "=" * w)
    print(f"  RUN {run_num:>2}  —  GA+SMOTE RESOURCE & PERFORMANCE")
    print("=" * w)
    print(f"  {'Test Set Size':<28}: {n_test:,} samples")
    print(f"  {'Features Used':<28}: {n_features}")
    print(f"  {'Total Inference Time':<28}: {inference_time:.4f} s")
    print(f"  {'Avg Time per Sample':<28}: {avg_time_per_sample:.8f} s")
    print(f"  {'Peak Memory Usage':<28}: {mem_used:.2f} MB")
    print(f"  {'CPU Utilization':<28}: {cpu_usage:.2f} %")
    print("-" * w)
    print(f"  {'Accuracy':<28}: {acc:.4f}")
    print(f"  {'Macro F1-Score':<28}: {f1_macro:.4f}")
    print(f"  {'Weighted F1-Score':<28}: {f1_weighted:.4f}")
    print("=" * w)
    print("  DETAILED CLASSIFICATION REPORT")
    print("=" * w)
    # Indent each line of the sklearn report
    for line in report.splitlines():
        print("  " + line)
    print("=" * w)



print("=" * 80)
print("BATCH EVALUATION: 15 GA+SMOTE MODELS WITH RESOURCE MONITORING")
print("=" * 80)

print("\n[1] Loading dataset...")
df = pd.read_csv(DATA_PATH)
df = df[df['label'] != 0].copy()
df['label'] = df['label'].map(label_merge_map)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
X = df.drop(columns=['label']).select_dtypes(include=['number'])
y = df['label']

X = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X), columns=X.columns)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"    Test size: {len(X_test):,}")


results = []
process = psutil.Process(os.getpid())

for run_num, (model_path, features_path) in enumerate(MODEL_FILES, start=1):
    print(f"\n>>> Loading Run {run_num} ({Path(model_path).name}) ...")

    try:
        # Load model + selected features
        model = joblib.load(model_path)
        with open(features_path, 'rb') as f:
            selected_features = list(pickle.load(f))

        X_test_sel = X_test[selected_features]

       
        mem_before = process.memory_info().rss / (1024 ** 2)
        cpu_before = psutil.cpu_percent(interval=None)   # prime the counter

        inference_start = time.perf_counter()
        y_pred = model.predict(X_test_sel)
        inference_time = time.perf_counter() - inference_start

        mem_after  = process.memory_info().rss / (1024 ** 2)
        mem_used   = max(mem_after - mem_before, 0.0)    # clamp negatives to 0
        cpu_usage  = psutil.cpu_percent(interval=None)

      
        acc         = accuracy_score(y_test, y_pred)
        f1_macro    = f1_score(y_test, y_pred, average='macro',    zero_division=0)
        f1_weighted = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        precision, recall, _, _ = precision_recall_fscore_support(
            y_test, y_pred, average=None, zero_division=0
        )
        weights            = np.bincount(y_test)
        precision_weighted = np.average(precision, weights=weights)
        recall_weighted    = np.average(recall,    weights=weights)

       
        print_run_report(
            run_num       = run_num,
            n_test        = len(X_test),
            n_features    = len(selected_features),
            inference_time= inference_time,
            mem_used      = mem_used,
            cpu_usage     = cpu_usage,
            acc           = acc,
            f1_macro      = f1_macro,
            f1_weighted   = f1_weighted,
            y_test        = y_test,
            y_pred        = y_pred,
            label_encoder = label_encoder,
        )

        results.append({
            'Run':               run_num,
            'Features':          len(selected_features),
            'Accuracy':          acc,
            'Precision (W)':     precision_weighted,
            'Recall (W)':        recall_weighted,
            'F1 (Macro)':        f1_macro,
            'F1 (Weighted)':     f1_weighted,
            'Inference Time (s)':inference_time,
            'Avg Time/Sample (s)':inference_time / len(X_test),
            'Memory (MB)':       mem_used,
            'CPU (%)':           cpu_usage,
        })

    except Exception as e:
        print(f"    ❌ Error in Run {run_num}: {e}")
        continue


if not results:
    print("\n❌ No models evaluated!")
    exit()

results_df = pd.DataFrame(results)

print("\n" + "=" * 80)
print("GA OPTIMIZATION STABILITY (15 RUNS) — SUMMARY TABLE")
print("=" * 80)
print(results_df.to_string(index=False,
      float_format=lambda x: f'{x:.4f}' if abs(x) < 1000 else f'{x:.2f}'))

print("\n" + "=" * 80)
print("AGGREGATE STATISTICS")
print("=" * 80)
print(f"{'Metric':<30} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 80)

for col in results_df.columns[1:]:
    m, s, lo, hi = (results_df[col].mean(), results_df[col].std(),
                    results_df[col].min(),  results_df[col].max())
    fmt = '.1f' if col == 'Features' else ('.4f' if m < 1000 else '.2f')
    print(f"{col:<30} {m:>10{fmt}} {s:>10{fmt}} {lo:>10{fmt}} {hi:>10{fmt}}")

print("\n" + "=" * 80)
print("COEFFICIENT OF VARIATION  (Lower = More Stable)")
print("=" * 80)
for col in ['Accuracy', 'F1 (Weighted)', 'F1 (Macro)']:
    cv = (results_df[col].std() / results_df[col].mean()) * 100
    print(f"  {col:<28} {cv:.4f}%")
print("=" * 80)

output_file = 'ga_stability_results_with_resources.csv'
results_df.to_csv(output_file, index=False)
print(f"\n💾 Results saved to: {output_file}")
print("\n✅ Batch evaluation complete!")

BATCH EVALUATION: 15 GA+SMOTE MODELS WITH RESOURCE MONITORING

[1] Loading dataset...
    Test size: 3,952

>>> Loading Run 1 (bestGAonly_rf_pipeline18.pkl) ...

  RUN  1  —  GA+SMOTE RESOURCE & PERFORMANCE
  Test Set Size               : 3,952 samples
  Features Used               : 23
  Total Inference Time        : 0.1078 s
  Avg Time per Sample         : 0.00002727 s
  Peak Memory Usage           : 8.69 MB
  CPU Utilization             : 75.60 %
------------------------------------------------------------------------
  Accuracy                    : 0.9704
  Macro F1-Score              : 0.9345
  Weighted F1-Score           : 0.9704
  DETAILED CLASSIFICATION REPORT
                precision    recall  f1-score   support
  
       Class_1     0.7070    0.6937    0.7003       160
       Class_2     1.0000    0.9938    0.9969       320
       Class_3     0.9969    1.0000    0.9984       320
       Class_4     0.8393    0.8812    0.8598       160
       Class_5     0.9454    0.9375    0

In [1]:
"""
Batch evaluation of 15 GA+SMOTE models with resource monitoring
Includes 0.9 confidence threshold — ambiguous samples excluded from metrics

Author: Tan Yi Feng
Student ID: 23WMR14766

"""

import pandas as pd
import numpy as np
import joblib
import pickle
import time
import psutil
import os
from sklearn.model_selection import train_test_split
from sklearn.metrics import (accuracy_score, f1_score,
                             precision_recall_fscore_support,
                             classification_report)
from sklearn.preprocessing import LabelEncoder
from sklearn.impute import SimpleImputer
from pathlib import Path
import sys

CONFIDENCE_THRESHOLD = 0.9


SCRIPT_DIR = Path.cwd()


DATA_PATH = SCRIPT_DIR / 'data' / 'FYP_Processed_Data' / 'balanced_train_data.csv'
MODELS_DIR = SCRIPT_DIR / 'data' / 'models'

MODEL_FILES = [
    (MODELS_DIR / 'best_rf_pipeline2.pkl', MODELS_DIR / 'selected_features2.pkl'),
    (MODELS_DIR / 'best_rf_pipeline3.pkl', MODELS_DIR / 'selected_features3.pkl'),
    (MODELS_DIR / 'best_rf_pipeline4.pkl', MODELS_DIR / 'selected_features4.pkl'),
    (MODELS_DIR / 'best_rf_pipeline5.pkl', MODELS_DIR / 'selected_features5.pkl'),
    (MODELS_DIR / 'best_rf_pipeline6.pkl', MODELS_DIR / 'selected_features6.pkl'),
    (MODELS_DIR / 'best_rf_pipeline7.pkl', MODELS_DIR / 'selected_features7.pkl'),
    (MODELS_DIR / 'best_rf_pipeline8.pkl', MODELS_DIR / 'selected_features8.pkl'),
    (MODELS_DIR / 'best_rf_pipeline9.pkl', MODELS_DIR / 'selected_features9.pkl'),
    (MODELS_DIR / 'best_rf_pipeline10.pkl', MODELS_DIR / 'selected_features10.pkl'),
    (MODELS_DIR / 'best_rf_pipeline11.pkl', MODELS_DIR / 'selected_features11.pkl'),
    (MODELS_DIR / 'best_rf_pipeline12.pkl', MODELS_DIR / 'selected_features12.pkl'),
    (MODELS_DIR / 'best_rf_pipeline13.pkl', MODELS_DIR / 'selected_features13.pkl'),
    (MODELS_DIR / 'best_rf_pipeline14.pkl', MODELS_DIR / 'selected_features14.pkl'),
    (MODELS_DIR / 'best_rf_pipeline15.pkl', MODELS_DIR / 'selected_features15.pkl'),
    (MODELS_DIR / 'best_rf_pipeline16.pkl', MODELS_DIR / 'selected_features16.pkl'),
]


label_merge_map = {
    1: 1, 2: 2, 3: 3, 4: 2, 5: 3,
    6: 4, 7: 5, 8: 6, 9: 5, 10: 5,
    11: 7, 15: 7, 12: 8, 16: 8,
    13: 9, 17: 9, 14: 10, 18: 10
}

CLASS_NAMES = {
    0: 'ARP Spoofing',
    1: 'MQTT Connect Flood',
    2: 'MQTT Publish Flood',
    3: 'MQTT Malformed',
    4: 'Reconnaissance',
    5: 'Recon (VulnScan)',
    6: 'ICMP Flood',
    7: 'SYN Flood',
    8: 'TCP Flood',
    9: 'UDP Flood',
}


def print_run_report(run_num, n_test, n_features, inference_time,
                     mem_used, cpu_usage,
                     # Full predictions (all samples)
                     acc_full, f1_macro_full, f1_weighted_full,
                     # High-confidence only
                     n_confident, n_ambiguous, ambiguous_rate,
                     acc_conf, f1_macro_conf, f1_weighted_conf,
                     y_test_conf, y_pred_conf,
                     label_encoder):

    avg_time_per_sample = inference_time / n_test
    classes      = label_encoder.classes_
    target_names = [CLASS_NAMES.get(c, f'Class_{c}') for c in classes]

    report = classification_report(
        y_test_conf, y_pred_conf,
        target_names=target_names,
        zero_division=0,
        digits=4
    )

    w = 72
    print("\n" + "=" * w)
    print(f"  RUN {run_num:>2}  —  GA+SMOTE RESOURCE & PERFORMANCE")
    print("=" * w)
    print(f"  {'Test Set Size':<30}: {n_test:,} samples")
    print(f"  {'Features Used':<30}: {n_features}")
    print(f"  {'Total Inference Time':<30}: {inference_time:.4f} s")
    print(f"  {'Avg Time per Sample':<30}: {avg_time_per_sample:.8f} s")
    print(f"  {'Peak Memory Usage':<30}: {mem_used:.2f} MB")
    print(f"  {'CPU Utilization':<30}: {cpu_usage:.2f} %")
    print("-" * w)
    print(f"  ALL SAMPLES (no threshold)")
    print(f"  {'Accuracy':<30}: {acc_full:.4f}")
    print(f"  {'Macro F1-Score':<30}: {f1_macro_full:.4f}")
    print(f"  {'Weighted F1-Score':<30}: {f1_weighted_full:.4f}")
    print("-" * w)
    print(f"  CONFIDENCE THRESHOLD = {CONFIDENCE_THRESHOLD}")
    print(f"  {'High-Confidence Samples':<30}: {n_confident:,}  ({100 - ambiguous_rate:.2f}%)")
    print(f"  {'Ambiguous Samples':<30}: {n_ambiguous:,}  ({ambiguous_rate:.2f}%)")
    print("-" * w)
    print(f"  HIGH-CONFIDENCE ONLY METRICS")
    print(f"  {'Accuracy':<30}: {acc_conf:.4f}")
    print(f"  {'Macro F1-Score':<30}: {f1_macro_conf:.4f}")
    print(f"  {'Weighted F1-Score':<30}: {f1_weighted_conf:.4f}")
    print("=" * w)
    print("  DETAILED CLASSIFICATION REPORT  (High-Confidence Samples Only)")
    print("=" * w)
    for line in report.splitlines():
        print("  " + line)
    print("=" * w)


print("=" * 80)
print("BATCH EVALUATION: 15 GA+SMOTE MODELS WITH CONFIDENCE THRESHOLD")
print(f"Confidence Threshold : {CONFIDENCE_THRESHOLD}")
print("=" * 80)

print("\n[1] Loading dataset...")
df = pd.read_csv(DATA_PATH)
df = df[df['label'] != 0].copy()
df['label'] = df['label'].map(label_merge_map)

df.replace([np.inf, -np.inf], np.nan, inplace=True)
X = df.drop(columns=['label']).select_dtypes(include=['number'])
y = df['label']

X = pd.DataFrame(SimpleImputer(strategy='median').fit_transform(X), columns=X.columns)

label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

print(f"    Test size: {len(X_test):,}")

results = []
process = psutil.Process(os.getpid())

for run_num, (model_path, features_path) in enumerate(MODEL_FILES, start=1):
    print(f"\n>>> Loading Run {run_num} ({Path(model_path).name}) ...")

    try:
       
        model = joblib.load(model_path)
        with open(features_path, 'rb') as f:
            selected_features = list(pickle.load(f))

        X_test_sel = X_test[selected_features]

        
        mem_before = process.memory_info().rss / (1024 ** 2)
        _          = psutil.cpu_percent(interval=None)  

        inference_start = time.perf_counter()
        y_pred      = model.predict(X_test_sel)
        y_proba     = model.predict_proba(X_test_sel)   
        inference_time = time.perf_counter() - inference_start

        mem_after  = process.memory_info().rss / (1024 ** 2)
        mem_used   = max(mem_after - mem_before, 0.0)
        cpu_usage  = psutil.cpu_percent(interval=None)

        
        acc_full         = accuracy_score(y_test, y_pred)
        f1_macro_full    = f1_score(y_test, y_pred, average='macro',    zero_division=0)
        f1_weighted_full = f1_score(y_test, y_pred, average='weighted', zero_division=0)

        precision_full, recall_full, _, _ = precision_recall_fscore_support(
            y_test, y_pred, average=None, zero_division=0
        )
        weights              = np.bincount(y_test)
        precision_w_full     = np.average(precision_full, weights=weights)
        recall_w_full        = np.average(recall_full,    weights=weights)

       
        max_proba       = y_proba.max(axis=1)             
        confident_mask  = max_proba >= CONFIDENCE_THRESHOLD
        ambiguous_mask  = ~confident_mask

        n_confident     = confident_mask.sum()
        n_ambiguous     = ambiguous_mask.sum()
        ambiguous_rate  = (n_ambiguous / len(y_test)) * 100

        y_test_conf     = y_test[confident_mask]
        y_pred_conf     = y_pred[confident_mask]

       
        acc_conf         = accuracy_score(y_test_conf, y_pred_conf)
        f1_macro_conf    = f1_score(y_test_conf, y_pred_conf, average='macro',    zero_division=0)
        f1_weighted_conf = f1_score(y_test_conf, y_pred_conf, average='weighted', zero_division=0)

        precision_conf, recall_conf, _, _ = precision_recall_fscore_support(
            y_test_conf, y_pred_conf, average=None, zero_division=0
        )
        weights_conf     = np.bincount(y_test_conf)
        precision_w_conf = np.average(precision_conf, weights=weights_conf)
        recall_w_conf    = np.average(recall_conf,    weights=weights_conf)

       
        print_run_report(
            run_num         = run_num,
            n_test          = len(X_test),
            n_features      = len(selected_features),
            inference_time  = inference_time,
            mem_used        = mem_used,
            cpu_usage       = cpu_usage,
            acc_full        = acc_full,
            f1_macro_full   = f1_macro_full,
            f1_weighted_full= f1_weighted_full,
            n_confident     = n_confident,
            n_ambiguous     = n_ambiguous,
            ambiguous_rate  = ambiguous_rate,
            acc_conf        = acc_conf,
            f1_macro_conf   = f1_macro_conf,
            f1_weighted_conf= f1_weighted_conf,
            y_test_conf     = y_test_conf,
            y_pred_conf     = y_pred_conf,
            label_encoder   = label_encoder,
        )

        results.append({
            'Run':                    run_num,
            'Features':               len(selected_features),
           
            'Accuracy (All)':         acc_full,
            'F1 Macro (All)':         f1_macro_full,
            'F1 Weighted (All)':      f1_weighted_full,
           
            'High-Conf Samples':      n_confident,
            'Ambiguous Samples':      n_ambiguous,
            'Ambiguous Rate (%)':     round(ambiguous_rate, 4),
            'High-Conf Rate (%)':     round(100 - ambiguous_rate, 4),
           
            'Accuracy (Conf)':        acc_conf,
            'F1 Macro (Conf)':        f1_macro_conf,
            'F1 Weighted (Conf)':     f1_weighted_conf,
            'Precision W (Conf)':     precision_w_conf,
            'Recall W (Conf)':        recall_w_conf,
          
            'Inference Time (s)':     inference_time,
            'Avg Time/Sample (s)':    inference_time / len(X_test),
            'Memory (MB)':            mem_used,
            'CPU (%)':                cpu_usage,
        })

    except Exception as e:
        print(f"    ❌ Error in Run {run_num}: {e}")
        continue


if not results:
    print("\n❌ No models evaluated!")
    exit()

results_df = pd.DataFrame(results)

print("\n" + "=" * 80)
print("GA OPTIMIZATION STABILITY (15 RUNS) — FULL SUMMARY")
print("=" * 80)
print(results_df.to_string(index=False,
      float_format=lambda x: f'{x:.4f}' if abs(x) < 1000 else f'{x:.2f}'))


print("\n" + "=" * 80)
print("THRESHOLD ANALYSIS RANKING  (sorted by High-Conf Rate)")
print("=" * 80)
rank_cols = ['Run', 'Features', 'High-Conf Rate (%)', 'Ambiguous Rate (%)',
             'High-Conf Samples', 'Ambiguous Samples',
             'Accuracy (Conf)', 'F1 Macro (Conf)', 'F1 Weighted (Conf)']
rank_df = results_df[rank_cols].sort_values('High-Conf Rate (%)', ascending=False)
print(rank_df.to_string(index=False,
      float_format=lambda x: f'{x:.4f}' if abs(x) < 1000 else f'{x:.2f}'))

print("\n" + "=" * 80)
print("BEST MODEL RECOMMENDATION")
print("=" * 80)

best = results_df.sort_values(
    ['High-Conf Rate (%)', 'F1 Weighted (Conf)'],
    ascending=[False, False]
).iloc[0]
print(f"  Best Run              : Run {int(best['Run'])}")
print(f"  Features              : {int(best['Features'])}")
print(f"  High-Confidence Rate  : {best['High-Conf Rate (%)']:.4f}%")
print(f"  Ambiguous Rate        : {best['Ambiguous Rate (%)']:.4f}%")
print(f"  Accuracy (Conf)       : {best['Accuracy (Conf)']:.4f}")
print(f"  F1 Weighted (Conf)    : {best['F1 Weighted (Conf)']:.4f}")
print(f"  F1 Macro (Conf)       : {best['F1 Macro (Conf)']:.4f}")
print("=" * 80)


print("\n" + "=" * 80)
print("AGGREGATE STATISTICS")
print("=" * 80)
print(f"{'Metric':<30} {'Mean':>10} {'Std':>10} {'Min':>10} {'Max':>10}")
print("-" * 80)
for col in results_df.columns[1:]:
    m  = results_df[col].mean()
    s  = results_df[col].std()
    lo = results_df[col].min()
    hi = results_df[col].max()
    fmt = '.1f' if col == 'Features' else ('.4f' if abs(m) < 1000 else '.2f')
    print(f"{col:<30} {m:>10{fmt}} {s:>10{fmt}} {lo:>10{fmt}} {hi:>10{fmt}}")


print("\n" + "=" * 80)
print("COEFFICIENT OF VARIATION  (Lower = More Stable)")
print("=" * 80)
for col in ['Accuracy (All)', 'F1 Weighted (All)', 'F1 Macro (All)',
            'Accuracy (Conf)', 'F1 Weighted (Conf)', 'F1 Macro (Conf)']:
    cv = (results_df[col].std() / results_df[col].mean()) * 100
    print(f"  {col:<30} {cv:.4f}%")
print("=" * 80)


output_file = 'ga_stability_threshold_results.csv'
results_df.to_csv(output_file, index=False)
print(f"\n💾 Results saved to: {output_file}")
print("\n✅ Batch evaluation with threshold analysis complete!")

BATCH EVALUATION: 15 GA+SMOTE MODELS WITH CONFIDENCE THRESHOLD
Confidence Threshold : 0.9

[1] Loading dataset...
    Test size: 3,952

>>> Loading Run 1 (best_rf_pipeline2.pkl) ...

  RUN  1  —  GA+SMOTE RESOURCE & PERFORMANCE
  Test Set Size                 : 3,952 samples
  Features Used                 : 25
  Total Inference Time          : 0.1693 s
  Avg Time per Sample           : 0.00004284 s
  Peak Memory Usage             : 0.83 MB
  CPU Utilization               : 47.40 %
------------------------------------------------------------------------
  ALL SAMPLES (no threshold)
  Accuracy                      : 0.9691
  Macro F1-Score                : 0.9318
  Weighted F1-Score             : 0.9693
------------------------------------------------------------------------
  CONFIDENCE THRESHOLD = 0.9
  High-Confidence Samples       : 3,696  (93.52%)
  Ambiguous Samples             : 256  (6.48%)
------------------------------------------------------------------------
  HIGH-CONFIDENC